# Perform Node Classification on the WACV Dataset

## Overview of the dataset
- The new dataset is in `./data/GraphPCB/Graph-W/graphs/` and `./data/GraphPCB/Graph-F/graphs/`
- The train/test sets contain 37/10 graphs each.
- The stats of the graphs are like this:
    - avg num of nodes in each graph are mostly around 50-600
    - avg num of node degrees are between 5.7-5.9
    - the diameter of the graph are a normal distribution centered at ~14
- The nodes have 4 types/categories:
    - 0-IC  the target
    - 1-DT (hard to distinguish from 0-IC)
    - 2-Diode (also hard to tell from 0-IC)
    - 3-Others (should be easier to distinguish)
- **Note that not all categories are presented in all the graphs (which means, there are some graphs that has less than 4 types of nodes)**

**Plans**
- Try some node classification models including GCN, GAT, and GraphSAGE

In [1]:
%load_ext autoreload
%autoreload 2

In [13]:
import sys, os
import numpy as np

import torch
import torch.nn as nn
from torch_geometric.data import Dataset
# Dataloader
from torch_geometric.loader import DataLoader

from utils.base_models import GCN, GAT, GraphSAGE, GIN, MLP
from utils.logger import PCB_Logger
from utils.utils import set_seed, compute_loss, compute_metrics

# Dataloaders

In [11]:
# Define a custom dataset to load graphs from the train/test directories
class GraphDataset(Dataset):
    def __init__(self, dataset_dir, transform=None):
        """
        Args:
            dataset_dir (str): Directory containing graph data files.
            transform (callable, optional): Transform to apply to each graph.
        """
        super().__init__()
        self.dataset_dir = dataset_dir
        self.graph_files = [f for f in os.listdir(dataset_dir) if f.endswith('.pt')]
        self.transform = transform

    def len(self):
        return len(self.graph_files)

    def get(self, idx):
        # Load torch-saved graph
        graph_path = os.path.join(self.dataset_dir, self.graph_files[idx])
        graph = torch.load(graph_path)

        # Apply optional transformations
        if self.transform:
            graph = self.transform(graph)

        return graph

    def get_all_labels(self):
        """
        Collect all node labels from the dataset for the sampler.
        Returns:
            Tensor: Concatenated node labels from all graphs.
        """
        all_labels = []
        for graph_file in self.graph_files:
            graph_path = os.path.join(self.dataset_dir, graph_file)
            graph = torch.load(graph_path)
            all_labels.append(graph.y) 
        return torch.cat(all_labels, dim=0)


def get_data_loaders(dataset_name, dataset_dir, batch_size=4):
    """
    Returns train and test DataLoaders for the specified dataset.

    Args:
        dataset_name (str): Name of the dataset ('wacv' or 'fpic').
        batch_size (int): Batch size for the DataLoaders.

    Returns:
        tuple: train_loader, test_loader
    """
    if dataset_name.lower() in ['wacv', 'fpic']:
        dataset_path = os.path.join(dataset_dir, f'Graph-{dataset_name[0].upper()}', 'graphs')
    elif dataset_name.lower() in ['fpic-knn', 'wacv-knn']:
        dataset_path = os.path.join(dataset_dir, f'Graph-{dataset_name[0].upper()}', 'KNN_graphs')
    elif dataset_name.lower() in ['fpic-cknn', 'wacv-cknn']:
        dataset_path = os.path.join(dataset_dir, f'Graph-{dataset_name[0].upper()}', 'Cosine_KNN_graphs')
    else:
        raise ValueError("Invalid dataset name. Choose either 'wacv' or 'fpic'.")

    train_dir = os.path.join(dataset_path, 'train')
    test_dir = os.path.join(dataset_path, 'test')

    train_dataset = GraphDataset(train_dir)
    test_dataset = GraphDataset(test_dir)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, test_loader


In [3]:
import glob
import pandas as pd

def generate_file_csvs(base_dirs):
    """
    For each dataset directory, generate a CSV listing .pt, image, and mask files.
    Args:
        base_dirs (dict): Mapping from dataset name to base directory path.
    """
    for name, base_dir in base_dirs.items():
        graph_dir = os.path.join(base_dir, "graphs")
        # Search for .pt files recursively under train and test subfolders in graph_dir
        pt_files = []
        for split in ["train", "test"]:
            split_dir = os.path.join(graph_dir, split)
            pt_files.extend(sorted(glob.glob(os.path.join(split_dir, "*.pt"), recursive=True)))
        print(f"Found {len(pt_files)} .pt files in {graph_dir} (train/test subfolders included)")
        
        image_dir = os.path.join(base_dir, "images")
        mask_dir = os.path.join(base_dir, "masks")

        rows = []
        for pt_path in pt_files:
            base_fname = os.path.splitext(os.path.basename(pt_path))[0]
            
            if name == "Graph-W":
                img_path = os.path.join(image_dir, base_fname + ".jpg")
                mask_path = os.path.join(mask_dir, base_fname + ".xml")
            else:
                img_path = os.path.join(image_dir, base_fname + ".png")
                mask_path = os.path.join(mask_dir, base_fname + ".png")
            rows.append({
            "pt_file": pt_path,
            "image_file": img_path,
            "mask_file": mask_path
            })
        df = pd.DataFrame(rows)
        csv_path = os.path.join(base_dir, f"{name}_file_list.csv")
        df.to_csv(csv_path, index=False)
        print(f"Saved: {csv_path}")

base_dirs = {
    "Graph-W": "./data/GraphPCB/Graph-W/",
    "Graph-F": "./data/GraphPCB/Graph-F/"
}

generate_file_csvs(base_dirs)

Found 47 .pt files in ./data/GraphPCB/Graph-W/graphs (train/test subfolders included)
Saved: ./data/GraphPCB/Graph-W/Graph-W_file_list.csv
Found 162 .pt files in ./data/GraphPCB/Graph-F/graphs (train/test subfolders included)
Saved: ./data/GraphPCB/Graph-F/Graph-F_file_list.csv


# Inference

In [4]:
def inference(model, test_loader):
    model.eval()
    all_preds, all_labels = [], []
    predictions = []

    with torch.no_grad():
        for graph_idx in range(len(test_loader.dataset)):
            device = next(model.parameters()).device
            graph = test_loader.dataset[graph_idx]
            graph_file_name = test_loader.dataset.graph_files[graph_idx]
            graph = graph.to(device)

            x, adj = graph.x, graph.edge_index
            y_true = graph.y
            # Forward pass
            logits = model(x, adj)
            preds = logits.argmax(dim=1)

            all_preds.append(preds.cpu())
            all_labels.append(y_true.cpu())
    
            predictions.append({
                "graph_id": graph_file_name,
                "labels": preds.tolist()
            })

    all_preds = torch.cat(all_preds, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    return all_preds.numpy(), all_labels.numpy(), predictions

# Training Function
- for every 10 epoch, save the model and evaluate on test set
- save the training log into a .txt
- save the last checkpoint
- save the predictions into a .json

In [5]:
def train_step(model, train_loader, optimizer, scheduler, device):
    total_loss = 0
    model.train()

    for batch in train_loader:
        batch = batch.to(device)

        batch = batch.sort(sort_by_row=False)

        x, adj = batch.x, batch.edge_index
        y = batch.y

        logits = model(x, adj)
    
        loss = compute_loss(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()
    avg_loss = total_loss / max(1, len(train_loader))
    return avg_loss


In [6]:
# the main function to train the model
def train_model(model, config):
    """
    Generic function to train different GNN models.
    """
    set_seed(config["seed"])
    train_loader, test_loader = get_data_loaders(config['dataset'], config['dataset_dir'], batch_size=config["batch_size"])

    logger = PCB_Logger(config=config)

    torch.cuda.empty_cache()

    # ✅ Move model to device
    model = model.to(config["device"])
    
    # ✅ Define optimizer & scheduler
    optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay'])
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=config["scheduler"]["step_size"], gamma=config["scheduler"]["gamma"])

    # ✅ Training Loop
    for epoch in range(config["num_epochs"]):
        avg_loss = train_step(model, train_loader, optimizer, scheduler, config["device"])
        logger.log(f"Epoch {epoch + 1:03d}, Loss: {avg_loss:.10f}")
        # ✅ Evaluate every 10 epochs OR at the last epoch
        if (epoch + 1) % 10 == 0 or (epoch + 1 == config["num_epochs"]):
            all_preds, all_labels, predictions = inference(model, test_loader)
            metrics = compute_metrics(all_preds, all_labels)
            logger.update_metrics(metrics, predictions)
    
    logger.finish_run()
    # save only the final model
    checkpoint_path = os.path.join(logger.checkpoint_dir, f"model_epoch_{epoch + 1}_{avg_loss:.4f}.pth")
    torch.save({
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'metrics': metrics,
    }, checkpoint_path)
    logger.log(f"✅ Model checkpoint saved to {checkpoint_path}")

    return model, metrics, logger.checkpoint_dir

# Basic Config Settings

In [7]:
# Set the home directory for the dataset
home_dir = os.path.join(os.path.expanduser("~"), "GraphPCB_Analysis")
dataset = "GraphPCB"
dataset_dir = os.path.join(home_dir, 'data', dataset)
print("Dataset directory:", dataset_dir)
# verify if the directory exists

Dataset directory: /home/lantian/GraphPCB_Analysis/data/GraphPCB


In [9]:
# Basic configuration 
config = {
    "experiment_name": "",
    "dataset_dir": dataset_dir,
    "home_dir": home_dir,
    "dataset": "wacv",
    "device": "cuda:0",
    "seed": 37,

    # model architecture
    "model": "",
    "input_dim": 1024,
    "hidden_dim": 256,
    "output_dim": 4,

    # regularization
    "dropout": 0.3,
    "use_batchnorm": True,
    "use_bias": False,
    "use_skip": True,
    "weight_decay": 1e-3,
    "scheduler": {"type": "StepLR", "step_size": 30, "gamma": 0.5},

    # training parameters
    "learning_rate": 0.001,
    "num_epochs": 200, 
}

# MLP

In [ ]:
run_num = 0
notes = ""
model_name = "MLP"
num_layers = 2
exp_name = f"{model_name}{num_layers}_{notes}_{run_num}"


config.update({
    "experiment_name": exp_name,
    "device": "cuda",
    "dataset": "wacv",
    "batch_size": 4,
    "model": model_name,
    "input_dim": 1024,
    "hidden_dim": 256,
    "output_dim": 4,
    "num_layers": num_layers,
    "dropout": 0.3,
    "use_batchnorm": True,
    "use_skip": False,
    "num_epochs": 200,
    "learning_rate": 0.0001,
    "weight_decay": 1e-3,
    "scheduler": {"type": "StepLR", "step_size": 10, "gamma": 0.5}
})

# Set seed before training
set_seed(42)

# Instantiate the MLP model
model = MLP(
    input_dim=config["input_dim"],
    hidden_dim=config["hidden_dim"],
    output_dim=config["output_dim"],
    num_layers=config["num_layers"],
    dropout=config["dropout"],
    use_batchnorm=config["use_batchnorm"],
    use_skip=config["use_skip"]
)
import time
start_time = time.time()
# Train the MLP model
train_model(model, config)
# Print the training time
end_time = time.time()
training_time = end_time - start_time
print(f"MLP training completed in {training_time:.2f} seconds.")


Checkpoint directory: /home/lantian/GraphPCB_Analysis/Graph-W-trained/MLP2-NLL_ImbalancedSampler_0
Experiment Configuration:
experiment_name: MLP2-NLL_ImbalancedSampler_0
dataset_dir: /home/lantian/GraphPCB_Analysis/data/GraphPCB
home_dir: /home/lantian/GraphPCB_Analysis
dataset: wacv
device: cuda
seed: 37
model: MLP
input_dim: 1024
hidden_dim: 256
output_dim: 4
dropout: 0.3
use_batchnorm: True
use_bias: False
use_skip: False
weight_decay: 0.001
scheduler: {'type': 'StepLR', 'step_size': 10, 'gamma': 0.5}
learning_rate: 0.0001
num_epochs: 200
batch_size: 4
num_layers: 2
loss_type: NLL
Using device: cuda
Loading dataset: wacv
Results will be saved to /home/lantian/GraphPCB_Analysis/Graph-W-trained/MLP2-NLL_ImbalancedSampler_0.
Epoch 001, Loss: 1.2752062082
Epoch 002, Loss: 1.0178039968
Epoch 003, Loss: 0.8839850664
Epoch 004, Loss: 0.8263472676
Epoch 005, Loss: 0.6981095850
Epoch 006, Loss: 0.6957672000
Epoch 007, Loss: 0.6111295104
Epoch 008, Loss: 0.6290513843
Epoch 009, Loss: 0.51589

# GCN

In [23]:
run_num = 0

config.update({
    "experiment_name": f"GCN_{run_num}",
    "dataset": "wacv",  # or "fpic"
    "batch_size": 4,
    "model": "GCN",

    "hidden_dim": 256,
    "num_layers": 2,
    "learning_rate": 0.001,
    
})

model = GCN(
    in_dim=config["input_dim"],
    hidden_dim=config["hidden_dim"],
    out_dim=config["output_dim"],
    num_layers=config["num_layers"],
    dropout=config["dropout"],
    use_batchnorm=config["use_batchnorm"],
)
print(config['dataset_dir'])

model, metrics, checkpoint_dir = train_model(model, config)

/home/lantian/GraphPCB_Analysis/data/GraphPCB
Checkpoint directory: /home/lantian/GraphPCB_Analysis/Graph-W-trained/GCN_0
Experiment Configuration:
experiment_name: GCN_0
dataset_dir: /home/lantian/GraphPCB_Analysis/data/GraphPCB
home_dir: /home/lantian/GraphPCB_Analysis
dataset: wacv
device: cuda
seed: 37
model: GCN
input_dim: 1024
hidden_dim: 256
output_dim: 4
dropout: 0.3
use_batchnorm: True
use_bias: False
use_skip: False
weight_decay: 0.001
scheduler: {'type': 'StepLR', 'step_size': 20, 'gamma': 0.5}
learning_rate: 0.001
num_epochs: 200
batch_size: 4
num_layers: 2
aggr: softmax
Using device: cuda
Loading dataset: wacv
Results will be saved to /home/lantian/GraphPCB_Analysis/Graph-W-trained/GCN_0.
Epoch 001, Loss: 1.5602668285
Epoch 002, Loss: 1.3860954940
Epoch 003, Loss: 1.2283719659
Epoch 004, Loss: 1.1456972420
Epoch 005, Loss: 1.0809451699
Epoch 006, Loss: 1.1787095368
Epoch 007, Loss: 1.0531815588
Epoch 008, Loss: 0.9992288947
Epoch 009, Loss: 0.9818951786
Epoch 010, Loss: 1.

# GAT

In [ ]:
run_num = 0


config.update({
    "experiment_name": f"GAT_{run_num}",
    
    "device": "cuda",
    "dataset": "fpic",  # or "fpic"
    "batch_size": 4,
    "model": "GAT",
    "input_dim": 1024,
    "hidden_dim": 1024,
    "output_dim": 4,
    "num_heads": 4,
    "num_layers": 2,
    "dropout": 0.3,
    "use_batchnorm": True,
    "use_bias": False,
    "num_epochs": 200,
    "learning_rate": 0.0001,
    "weight_decay": 1e-3,
    "scheduler": {"type": "StepLR", "step_size": 20, "gamma": 0.5}
})


# Instantiate the model
model = GAT(
    in_dim=config["input_dim"],
    hidden_dim=config["hidden_dim"],
    out_dim=config["output_dim"],
    num_layers=config["num_layers"],
    num_heads=config["num_heads"],
    dropout=config["dropout"],
    use_batchnorm=config["use_batchnorm"]
)


model, metrics, checkpoint_dir = train_model(model, config)

Checkpoint directory: /home/lantian/PCB_Analysis/FPIC-trained/GAT-NLL_0
Experiment Configuration:
experiment_name: GAT-NLL_0
dataset_dir: /home/lantian/GraphPCB/
home_dir: /home/lantian/
dataset: fpic
device: cuda
model: GAT
input_dim: 1024
hidden_dim: 1024
output_dim: 4
dropout: 0.3
use_batchnorm: True
use_bias: False
weight_decay: 0.001
scheduler: {'type': 'StepLR', 'step_size': 20, 'gamma': 0.5}
learning_rate: 0.0001
num_epochs: 200
batch_size: 4
num_layers: 2
loss_type: NLL
num_heads: 4
Using device: cuda
Loading dataset: fpic
Results will be saved to /home/lantian/PCB_Analysis/FPIC-trained/GAT-NLL_0.
Epoch 001, Loss: 1.3706389912
Epoch 002, Loss: 1.2006024862
Epoch 003, Loss: 1.1642258969
Epoch 004, Loss: 1.0859312477
Epoch 005, Loss: 1.0945229880
Epoch 006, Loss: 1.0387999539
Epoch 007, Loss: 1.0586524893
Epoch 008, Loss: 1.0353469273
Epoch 009, Loss: 1.0305183180
Epoch 010, Loss: 0.9872889848
  F1-Score (macro): 0.2718814467
  Weighted F1: 0.6093899855
  Subset F1-Score (3-class

# GIN

In [ ]:

run_num = 0


config.update({
    "experiment_name": "GIN-{}".format(run_num),
    "device": "cuda",
    "dataset": "wacv",  # or "fpic"
    "batch_size": 4,
    "model": "GIN",
    "input_dim": 1024,
    "hidden_dim": 256,
    "output_dim": 4,
    "num_layers": 2,
    "dropout": 0.3,
    "use_batchnorm": True,
    "use_bias": False,
    "use_skip": False,
    "num_epochs": 200,
    "learning_rate": 0.0001,
    "weight_decay": 1e-3,
    "scheduler": {"type": "StepLR", "step_size": 20, "gamma": 0.5}
})

# Set seed before training
set_seed(42)

model = GIN(
    in_dim=config["input_dim"],
    hidden_dim=config["hidden_dim"],
    out_dim=config["output_dim"],
    num_layers=config["num_layers"],
    dropout=config["dropout"],
    use_batchnorm=config["use_batchnorm"],
    use_skip=config["use_skip"]
)

model, metrics, checkpoint_dir = train_model(model, config)


Checkpoint directory: /home/lantian/GraphPCB_Analysis/Graph-W-trained/GIN-0
Experiment Configuration:
experiment_name: GIN-0
dataset_dir: /home/lantian/GraphPCB_Analysis/data/GraphPCB
home_dir: /home/lantian/GraphPCB_Analysis
dataset: wacv
device: cuda
seed: 37
model: GIN
input_dim: 1024
hidden_dim: 256
output_dim: 4
dropout: 0.3
use_batchnorm: True
use_bias: False
use_skip: False
weight_decay: 0.001
scheduler: {'type': 'StepLR', 'step_size': 20, 'gamma': 0.5}
learning_rate: 0.0001
num_epochs: 200
batch_size: 4
num_layers: 2
aggr: softmax
Using device: cuda
Loading dataset: wacv
Results will be saved to /home/lantian/GraphPCB_Analysis/Graph-W-trained/GIN-0.
Epoch 001, Loss: 1.4722096205
Epoch 002, Loss: 1.2252021313
Epoch 003, Loss: 1.0660099745
Epoch 004, Loss: 1.0015074313
Epoch 005, Loss: 0.8710965455
Epoch 006, Loss: 0.8534360230
Epoch 007, Loss: 0.8500327885
Epoch 008, Loss: 0.7572288990
Epoch 009, Loss: 0.7307833672
Epoch 010, Loss: 0.6739200056
  F1-Score (macro): 0.2140246554
 

/home/lantian/GraphPCB_Analysis/utils/utils.py:83: RuntimeWarning: invalid value encountered in divide
  precision = np.mean(TP / (TP + FP)) if np.sum(TP + FP) > 0 else 0


Epoch 034, Loss: 0.2941749796
Epoch 035, Loss: 0.2904956073
Epoch 036, Loss: 0.2581216872
Epoch 037, Loss: 0.2306968853
Epoch 038, Loss: 0.2823069736
Epoch 039, Loss: 0.2633401915
Epoch 040, Loss: 0.2475876957
  F1-Score (macro): 0.3279116549
  Weighted F1: 0.8732346085
  Subset F1-Score (3-class): 0.6445024227
  F1 per class: [0.28180039138943247, 0.0893854748603352, 0.039603960396039604, 0.9008567931456547]
  Precision per class: [0.1935483870967742, 0.04838709677419355, 0.025974025974025976, 0.9818093621149648]
  Recall per class: [0.5179856115107914, 0.5853658536585366, 0.08333333333333333, 0.8322368421052632]
  Confusion Matrix:
[[72, 15, 1, 51], [6, 24, 0, 11], [2, 7, 2, 13], [292, 450, 74, 4048]]
Epoch 041, Loss: 0.2409873188
Epoch 042, Loss: 0.2347062856
Epoch 043, Loss: 0.2407587335
Epoch 044, Loss: 0.2273192801
Epoch 045, Loss: 0.2492783502
Epoch 046, Loss: 0.2206832349
Epoch 047, Loss: 0.2382278800
Epoch 048, Loss: 0.2340003453
Epoch 049, Loss: 0.2199783370
Epoch 050, Loss: 

# GraphSAGE

## Aggregators

In [19]:
def get_aggregator(aggr_type):
    """
    Get the appropriate aggregator function based on the type.

    Args:
        aggr_type (str): Type of aggregation ('mean', 'sum', 'max').

    Returns:
        str: Aggregator type.
    """
    if aggr_type == 'softmax':
        return aggr.SoftmaxAggregation(learn=True)
    elif aggr_type == 'attn':
        gate_nn = nn.Sequential(
                        nn.Linear(1024, 128),
                        nn.ReLU(),
                        nn.Linear(128, 1)
                    )
        return aggr.AttentionalAggregation(gate_nn=gate_nn)
    else:
        return aggr_type

In [ ]:
from torch_geometric.nn import aggr
run_num = 1
notes = ""
model_name = "GraphSAGE"
num_layers = 2
aggr_type = "softmax" # "mean" | "max" |"sum" | 'std' | 'attn'
# https://pytorch-geometric.readthedocs.io/en/latest/modules/nn.html?highlight=torch_geometric+nn+aggr#aggregation-operators
# multi_aggr_type = aggr.MultiAggregation(['mean', 'std', aggr.SoftmaxAggregation(learn=True)])


config.update({
    "experiment_name": f"{model_name}{num_layers}_{aggr_type}_{notes}_{run_num}",
    "device": "cuda",
    "dataset": "wacv",  # or "fpic"
    "batch_size": 8,

    "model": model_name,
    "input_dim": 1024,
    "hidden_dim": 1024,
    "output_dim": 4,
    "num_layers": num_layers,
    "dropout": 0.3,
    "use_batchnorm": True,
    "use_skip": False,
    "aggr": aggr_type,  # Aggregation type
    "num_epochs": 200,
    "learning_rate": 0.0001,
    "weight_decay": 1e-3,
    "scheduler": {"type": "StepLR", "step_size": 10, "gamma": 0.5}
})

gate_nn = nn.Sequential(
    nn.Linear(1024, 128),
    nn.ReLU(),
    nn.Linear(128, 1)
)

# ✅ Instantiate the 2-layer GraphSAGE model
model = GraphSAGE(
    in_dim=config["input_dim"],
    hidden_dim=config["hidden_dim"],
    out_dim=config["output_dim"],
    num_layers=config["num_layers"],
    dropout=config["dropout"],
    use_batchnorm=config["use_batchnorm"],
    aggr=get_aggregator(config["aggr"]),
    use_skip=config["use_skip"]
)

model, metrics, checkpoint_dir = train_model(model, config)


Checkpoint directory: /home/lantian/GraphPCB_Analysis/Graph-W-trained/knn-GraphSAGE2_softmax_knn_1
Experiment Configuration:
experiment_name: knn-GraphSAGE2_softmax_knn_1
dataset_dir: /home/lantian/GraphPCB_Analysis/data/GraphPCB
home_dir: /home/lantian/GraphPCB_Analysis
dataset: wacv
device: cuda
seed: 37
model: GraphSAGE
input_dim: 1024
hidden_dim: 1024
output_dim: 4
dropout: 0.3
use_batchnorm: True
use_bias: False
use_skip: False
weight_decay: 0.001
scheduler: {'type': 'StepLR', 'step_size': 10, 'gamma': 0.5}
learning_rate: 0.0001
num_epochs: 200
batch_size: 8
num_layers: 2
aggr: softmax
Using device: cuda
Loading dataset: wacv
Results will be saved to /home/lantian/GraphPCB_Analysis/Graph-W-trained/knn-GraphSAGE2_softmax_knn_1.
Epoch 001, Loss: 1.4074495077
Epoch 002, Loss: 0.9720222473
Epoch 003, Loss: 0.6464896441
Epoch 004, Loss: 0.5200471997
Epoch 005, Loss: 0.4182893932
Epoch 006, Loss: 0.3378563225
Epoch 007, Loss: 0.2647712857
Epoch 008, Loss: 0.2143421948
Epoch 009, Loss: 0